In [31]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [32]:
data_dir = r"E:\AI Track\ITI\Deep learning\Day-2"

train_ds = image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=123,
    image_size=(224, 224), batch_size=32, label_mode='binary'
)
val_ds = image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=123,
    image_size=(224, 224), batch_size=32, label_mode='binary'
)

Found 253 files belonging to 2 classes.


Using 203 files for training.
Found 253 files belonging to 2 classes.
Using 50 files for validation.


In [33]:
train_images, train_labels = tuple(zip(*train_ds.unbatch().as_numpy_iterator()))
test_images, test_labels = tuple(zip(*val_ds.unbatch().as_numpy_iterator()))

train_images = np.array(train_images) / 255.0
train_labels = np.array(train_labels)
test_images = np.array(test_images) / 255.0
test_labels = np.array(test_labels)

print(train_images.shape)
print(test_images.shape)


(203, 224, 224, 3)
(50, 224, 224, 3)


In [34]:
inpt_dim = (224, 224, 3)
inpt_img = Input(shape=inpt_dim)

c11 = Conv2D(32, (3, 3), activation='relu')(inpt_img)
p12 = MaxPooling2D(pool_size=(2, 2))(c11)
bn13 = BatchNormalization()(p12)

c14 = Conv2D(64, (3, 3), activation='relu')(bn13)
p15 = MaxPooling2D(pool_size=(2, 2))(c14)
bn16 = BatchNormalization()(p15)

f17 = Flatten()(bn16)
do18 = Dropout(0.5)(f17)
d19 = Dense(units=128, activation='relu')(do18)
do110 = Dropout(0.2)(d19)
d111 = Dense(units=64, activation='relu')(do110)
do112 = Dropout(0.1)(d111)

output = Dense(units=1, activation='sigmoid')(do112)
classifier = Model(inpt_img, output)

opt = RMSprop(learning_rate=0.001)
classifier.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
print(classifier.summary())


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 111, 111, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 54, 54, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,916,097 (91.23 MB)

 Trainable params: 23,915,905 (91.23 MB)

 Non-trainable params: 192 (768.00 B)

None


In [35]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_delta=1e-4, mode='min', verbose=1
)
stop_alg = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)
checkpoint_filepath = 'cnn_model.h5'
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath, monitor='val_accuracy', mode='max', save_best_only=True
)

hist = classifier.fit(
    train_images, train_labels,
    batch_size=16,
    epochs=10,
    callbacks=[stop_alg, reduce_lr, model_checkpoint],
    shuffle=True,
    validation_data=(test_images, test_labels)
)

# Save and load weights
classifier.save_weights("cnn1.weights.h5")
classifier.load_weights('cnn1.weights.h5')

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.5911 - loss: 30.6132

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 210ms/step - accuracy: 0.5911 - loss: 30.6132 - val_accuracy: 0.7600 - val_loss: 0.7763 - learning_rate: 0.0010
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.7734 - loss: 3.9255

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 197ms/step - accuracy: 0.7734 - loss: 3.9255 - val_accuracy: 0.8200 - val_loss: 0.5242 - learning_rate: 0.0010
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 184ms/step - accuracy: 0.7635 - loss: 5.7411 - val_accuracy: 0.7800 - val_loss: 0.6139 - learning_rate: 0.0010
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 180ms/step - accuracy: 0.8079 - loss: 3.0883 - val_accuracy: 0.7200 - val_loss: 0.6635 - learning_rate: 0.0010
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 167ms/step - accuracy: 0.7980 - loss: 3.1386 - val_accuracy: 0.3600 - val_loss: 4.5582 - learning_rate: 0.0010
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 166ms/step - accuracy: 0.8916 - loss: 1.3391 - val_accuracy: 0.3600 - val_loss: 6.1697 - learning_rate: 0.0010
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.8818 - loss: 1.7891
Epoch 7: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 173ms/step - accuracy: 0.8818 - loss: 1.7891 - val_accuracy

In [36]:
y_hat = classifier.predict(test_images)
y_pred = (y_hat > 0.5).astype(int)

accuracy = np.mean(y_pred == test_labels)
print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")

print("\n Classification Report")
print(classification_report(test_labels, y_pred, target_names=['no', 'yes']))



2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step

Final Test Accuracy: 80.00%

 Classification Report
              precision    recall  f1-score   support

          no       0.75      0.67      0.71        18
         yes       0.82      0.88      0.85        32

    accuracy                           0.80        50
   macro avg       0.79      0.77      0.78        50
weighted avg       0.80      0.80      0.80        50

